# Regressão Logística — contrato, custo de ajuste e espaço de busca

Este notebook é a frente de **modelos interpretáveis** da comparação da Seção 4.4 (ART.7). Ele
começa onde o `modelagem.ipynb` da Seção 4.3 termina: consome o contrato de dados já pronto e não
repete nenhuma etapa de limpeza, integração ou pré-processamento.

Princípios adotados, os mesmos do notebook da Seção 4.3:

- consumir a matriz do contrato, sem reprocessar as fontes brutas nem reajustar transformador;
- importar de `src/` em vez de colar código na célula, para que notebook e `pytest` exercitem
  exatamente o mesmo módulo;
- nenhuma partição criada aqui. A divisão por data e por Cliente já veio congelada, e recriá-la
  neste notebook produziria uma segunda divisão que ninguém acordou.

**Seção 1, entregue pelo card #206.** Mede quanto custa um ajuste da Regressão Logística sobre o
contrato e se ela converge. Os dois números existem para dimensionar a grade de `GridSearchCV`
do card #207 antes de a busca começar, e não para avaliar qualidade: nenhuma métrica de
desempenho sai desta seção.

## 1. Custo de um ajuste e convergência (card #206)

### 1.1. Dependências

As versões de `scikit-learn` e `scipy` estão fixas em `requirements.txt` a partir do Sprint 04.
A fixação não é burocracia: o número de iterações até convergir, medido na Seção 1.4, depende da
implementação do solver, e um piso `>=` deixaria o Colab instalar outra minor e devolver um custo
diferente do que a grade do card #207 assume.

In [ ]:
!pip install -q -r requirements.txt

import sys
import time
from pathlib import Path

import pandas as pd
import scipy
import sklearn

# Inclui src/ no caminho de importação tanto no projeto local quanto no Colab.
# O break evita que duas cópias dos módulos fiquem no caminho ao mesmo tempo, o
# que tornaria ambíguo qual delas o import resolve.
for pasta_codigo in [Path("src"), Path("../src"), Path("/content/src")]:
    if pasta_codigo.exists():
        sys.path.insert(0, str(pasta_codigo.resolve()))
        break

from fumaca_logistica import medir_ajuste_unico, menor_max_iter_que_converge
from matriz import preparar_matriz, resumo_da_matriz

print("scikit-learn", sklearn.__version__)
print("scipy", scipy.__version__)

As versões impressas têm que ser `scikit-learn 1.9.1` e `scipy 1.18.1`, as mesmas declaradas em
`requirements.txt`. Divergência aqui invalida a comparação de custo das seções seguintes: o tempo
medido passaria a descrever outra implementação do `lbfgs`, e a conta que dimensiona a grade do
card #207 deixaria de valer.

### 1.2. Carga da base analítica

A busca cobre o projeto local e o Colab, com e sem Drive montado. Se nenhum caminho existir, a
célula falha com a instrução do que fazer, em vez de seguir com uma base vazia. A base não é
versionada, conforme o Termo de Abertura, que veda publicar dados do parceiro.

In [ ]:
# No Colab, monte o Drive antes de rodar esta célula:
#   from google.colab import drive; drive.mount('/content/drive')

CAMINHOS_POSSIVEIS = [
    Path("data/processed/base_analitica.parquet"),
    Path("../data/processed/base_analitica.parquet"),
    Path("/content/data/processed/base_analitica.parquet"),
    Path("/content/drive/MyDrive/projeto-azul/data/processed/base_analitica.parquet"),
]

caminho_base = next((c for c in CAMINHOS_POSSIVEIS if c.exists()), None)
if caminho_base is None:
    raise FileNotFoundError(
        "base_analitica.parquet não encontrada. Rode notebooks/pre-processamento.ipynb "
        f"ou ajuste o caminho. Procurado em: {[str(c) for c in CAMINHOS_POSSIVEIS]}"
    )

df = pd.read_parquet(caminho_base)
print(f"{caminho_base}: {df.shape[0]:,} linhas, {df.shape[1]} colunas")
print(f"Clientes distintos: {df['ID_GOLDENRECORD'].nunique():,}")

A base tem **484.915 linhas e 51 colunas**, com **407.139 valores distintos de
`ID_GOLDENRECORD`**. A distância entre os dois números é o motivo de a validação cruzada do card
#210 precisar agrupar por Cliente: há Cliente com mais de uma resposta, e a Hipótese 4 da Seção
4.2.4 mostrou que quem detratou uma vez volta a detratar com chance 4,61 vezes maior. Um Cliente
presente em dois folds faria o modelo reconhecer a pessoa em vez de aprender o fenômeno.

### 1.3. Matriz do contrato

`preparar_matriz`, de `src/matriz.py`, executa numa chamada as três etapas que precisam acontecer
nesta ordem: particionar por data e por Cliente, selecionar a allowlist de features do score
pós-viagem e ajustar o pré-processador **apenas no treino**.

Esta é a única fonte de `X`, `y`, `grupos` e do transformador em todo o notebook. As datas de
corte vêm do registro de decisão do #127, não são escolhidas aqui.

In [ ]:
CORTE_VALIDACAO = "2025-07-01"
CORTE_TESTE = "2026-01-01"

inicio = time.perf_counter()
preparo = preparar_matriz(df, corte_validacao=CORTE_VALIDACAO, corte_teste=CORTE_TESTE)
print(f"preparar_matriz: {time.perf_counter() - inicio:.1f}s")

matriz_treino = preparo["matrizes"]["treino"]
alvo_treino = preparo["y"]["treino"]
grupos_treino = preparo["grupos"]["treino"]

print(f"matriz de treino: {matriz_treino.shape[0]:,} linhas x {matriz_treino.shape[1]} colunas")
print(f"colunas na saída do pré-processador: {len(preparo['preprocessador'].get_feature_names_out())}")
print(f"Clientes no treino: {grupos_treino.nunique():,}")
print(f"prevalência de Detrator no treino: {alvo_treino.mean():.4f}")

resumo_da_matriz(preparo)

O contrato entrega uma matriz de treino com **341.962 linhas e 38 colunas**, cobrindo **307.510
Clientes**, e a prevalência de Detrator no treino é de **0,2043**. As 38 colunas da matriz são
exatamente as 38 que `get_feature_names_out` reporta, o que confirma que nada foi transformado
fora do contrato. O preparo inteiro custa cerca de **5,3 segundos**, desprezível diante do ajuste
medido a seguir.

A prevalência de 0,2043 é o número que o card #207 usa para justificar o eixo `class_weight` da
grade: com quatro não Detratores para cada Detrator, o modelo sem reponderação tende a acertar a
classe majoritária e a errar justamente quem a operação precisa encontrar.

### 1.4. Ajuste único: quanto custa e se converge

`medir_ajuste_unico` ajusta `LogisticRegression` **uma vez** sobre a matriz do contrato e devolve
tempo, iterações e convergência. Ele não monta `Pipeline`, que é o card #208, não reparticiona e
não calcula métrica: o número que interessa aqui é o relógio.

O aviso de convergência é capturado em vez de silenciado. Um ajuste que para no limite de
iterações entrega coeficiente provisório, e o card #212 lê exatamente esses coeficientes como
odds ratio.

In [ ]:
medicao_padrao = medir_ajuste_unico(matriz_treino, alvo_treino)

for chave in ("max_iter", "solver", "n_iter", "convergiu", "segundos"):
    print(f"{chave}: {medicao_padrao[chave]}")

if medicao_padrao["aviso"]:
    print("\naviso de convergência:")
    print(medicao_padrao["aviso"])

**O achado do card: a Regressão Logística não converge no `max_iter` padrão.** Com
`max_iter=100`, `solver='lbfgs'` e semente 42, o ajuste consome as 100 iterações, leva cerca de
**8,0 segundos** e emite `ConvergenceWarning` com `STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT`.

**A escala das features não é a causa, e isso importa.** O aviso do scikit-learn sugere escalonar
os dados, mas o contrato já faz isso: `matriz._montar_preprocessador` aplica `RobustScaler` nas
numéricas e codificação one-hot nas categóricas, então o `lbfgs` já recebe colunas em ordem de
grandeza comparável. Montar um escalonamento novo por fora do contrato para tentar acelerar a
convergência produziria uma segunda matriz e quebraria a comparação da Seção 4.4, que exige que
os quatro modelos consumam a mesma. O caminho correto é subir o `max_iter`, que é o que a Seção
1.5 mede.

A consequência para o card #207 é direta: `max_iter` não pode ficar no default na grade de busca,
senão toda combinação testada seria avaliada com coeficiente truncado.

### 1.5. Qual `max_iter` basta, e quanto ele custa

`menor_max_iter_que_converge` sobe o limite pela escala 100, 200, 400, 800, 1600 e para na
primeira tentativa que converge. Dobrar a cada passo mantém o custo total de procurar da ordem do
último ajuste, em vez da soma de uma varredura fina.

A trilha de tentativas é registrada de propósito: **só o tempo do ajuste que converge entra na
conta de custo da grade do card #207.** Os ajustes truncados param antes e custam menos, e usá-los
subestimaria a busca.

In [ ]:
busca_max_iter = menor_max_iter_que_converge(matriz_treino, alvo_treino)

for tentativa in busca_max_iter["tentativas"]:
    estado = "converge" if tentativa["convergiu"] else "trunca"
    print(f"max_iter={tentativa['max_iter']:>5} | n_iter={tentativa['n_iter']:>5} | "
          f"{tentativa['segundos']:>6.1f}s | {estado}")

escolhido = busca_max_iter["escolhido"]
print(f"\nescolhido: max_iter={escolhido['max_iter']}, "
      f"converge em {escolhido['n_iter']} iterações, {escolhido['segundos']:.1f}s")

A escala mostra onde está o limite:

| `max_iter` | iterações usadas | tempo | resultado |
|---:|---:|---:|---|
| 100 | 100 | 9,2s | trunca |
| 200 | 200 | 18,2s | trunca |
| 400 | 400 | 36,5s | trunca |
| 800 | **534** | **47,4s** | **converge** |

**A Regressão Logística converge em 534 iterações e custa cerca de 47 segundos por ajuste**, medido
nesta máquina (WSL2 local, CPU, scikit-learn 1.9.1 e scipy 1.18.1). No Colab o número muda com a
máquina sorteada, e por isso o que se transporta para o card #207 não é o segundo exato e sim a
ordem de grandeza: **um ajuste que converge custa dezenas de segundos, não centenas de
milissegundos.**

Até 400 iterações o tempo cresce proporcionalmente ao limite, porque o ajuste é truncado e gasta
todas as iterações disponíveis. Em 800 ele para antes, em 534, e é por isso que o tempo não dobra
de novo.

**O que este card entrega ao card #207**, que é quem declara a grade:

- `max_iter` precisa entrar fixado em pelo menos **800** na grade, e não como default;
- a conta de custo da busca é **47s por ajuste x número de folds x número de combinações**. Com os
  5 folds da validação por Cliente, cada combinação custa cerca de 4 minutos em série, o que
  limita a grade a poucas dezenas de combinações dentro de uma sessão do Colab;
- a prevalência de **0,2043** é o dado que sustenta o eixo `class_weight` da grade.

O mesmo número vai para a Luiza, que precisa dele para dimensionar a grade da Árvore de Decisão
sob o mesmo critério.

## 2. Espaço de busca da Regressão Logística (card #207)

Esta seção declara a grade que o card #210 vai executar no `GridSearchCV`, **antes** de qualquer
busca rodar. A ordem importa para o barema: grade declarada e justificada antes da execução é
evidência de método, enquanto grade ajustada depois de ver o resultado é tentativa e erro contada
como se fosse método.

A grade vive em `src/espaco_busca_logistica.py`, não numa célula daqui. Assim o `pytest` confere
que toda combinação declarada é válida, e o card #210 importa exatamente o mesmo objeto em vez de
reescrevê-lo.

### 2.1. O que a grade busca, e o que fica fixo

Três eixos variam. Dois parâmetros ficam fixos, e essa também é uma decisão do card:

| Parâmetro | Valores | Por que |
|---|---|---|
| `C` | 0.01, 0.1, 1.0, 10.0 | Escala logarítmica, de regularização forte a fraca. São 38 colunas para 341.962 linhas de treino, regime em que a verossimilhança domina a penalidade: quatro pontos separados por uma ordem de grandeza mostram se `C` tem efeito, sem gastar folds na terceira casa decimal |
| `l1_ratio` | 0.0, 1.0 | L2 pura contra L1 pura. A L1 zera coeficiente, e coeficiente zerado sai da leitura de odds ratio do card #212 |
| `class_weight` | `None`, `"balanced"` | A prevalência de Detrator no treino é 0,2043, quatro não Detratores para cada Detrator |
| `solver` | fixo: `lbfgs` na L2, `liblinear` na L1 | **Não é eixo de busca** |
| `max_iter` | fixo em 1600 | **Não é eixo de busca** |

**Por que `penalty` não aparece.** Ele está depreciado desde o scikit-learn 1.8 e é removido na
1.10. Quem expressa a penalidade agora é `l1_ratio`, com `0.0` para L2 e `1.0` para L1. Declarar a
grade com o parâmetro antigo faria a busca inteira emitir `FutureWarning` e quebraria na próxima
atualização do `requirements.txt`.

**Por que `solver` não é eixo de busca.** A verossimilhança da Regressão Logística com penalidade
L1 ou L2 é convexa: dois solvers que convergem chegam ao mesmo ótimo. O que muda entre eles é o
tempo, não o modelo. Buscar sobre o solver gastaria folds para escolher entre respostas iguais.
Ele fica fixo em cada subgrade, escolhido pela penalidade que aquela subgrade usa.

**Por que `max_iter` não é eixo de busca.** Buscar sobre o limite de iterações compararia quem
parou antes com quem terminou, e o card #212 lê esses coeficientes como odds ratio. O valor fixo
vem da medição da Seção 2.3.

**Por que a grade é uma lista de subgrades.** Nem toda combinação de `l1_ratio` com `solver`
existe: `lbfgs` só aceita L2 e levanta `ValueError` com L1. Um produto cartesiano único geraria
esse par inválido, e a busca morreria no meio depois de já ter gasto folds. A lista de subgrades é
como o `GridSearchCV` recebe espaços em que os eixos não são independentes.

In [ ]:
import pandas as pd

from espaco_busca_logistica import (
    GRADE_LOGISTICA,
    MAX_ITER,
    SEGUNDOS_POR_AJUSTE,
    combinacoes,
    custo_estimado,
    medir_ajuste,
)
from validacao import N_FOLDS

tabela_grade = pd.DataFrame(combinacoes())
print(f"{len(tabela_grade)} combinações, {N_FOLDS} folds, max_iter fixo em {MAX_ITER}")
tabela_grade.groupby(["solver", "l1_ratio"], dropna=False).size().rename("combinações")

São **16 combinações**, oito em cada subgrade: 4 valores de `C` x 2 de `class_weight`. O número de
folds vem de `validacao.N_FOLDS`, que é o `GroupKFold` de cinco folds por Cliente já usado na
Seção 4.3, e não um valor escolhido aqui.

### 2.2. `class_weight` e a prevalência de 0,2043

Com quatro não Detratores para cada Detrator, um modelo sem reponderação tende a acertar a classe
majoritária: ele erra pouco no total justamente porque quase nunca aponta Detrator, que é quem a
operação precisa encontrar antes do voo. `class_weight="balanced"` reponderar na proporção inversa
da frequência e empurra o modelo na direção oposta, achando mais Detratores ao custo de mais falso
positivo.

Os dois erros não custam o mesmo para a Azul, e por isso o eixo é medido em vez de decidido no
padrão da biblioteca: a escolha final sai da métrica oficial da comparação, no card #210, não
deste card.

### 2.3. Quanto custa a busca

A conta que dimensiona a grade é **combinações x folds x tempo de um ajuste**, e o tempo precisa
ser medido, não estimado de catálogo. O card #206 mediu um ajuste só, no ponto de partida da
biblioteca (`C=1.0`, sem reponderação): 47 s e 534 iterações. Esse número não cobre a grade, porque
o tempo muda com `C` e com `class_weight`.

A célula abaixo mede os **extremos** de cada subgrade, que é o que dá teto à conta.

In [ ]:
piores_casos = [
    {"C": C, "l1_ratio": l1, "class_weight": peso, "solver": solver, "max_iter": MAX_ITER}
    for solver, l1 in (("lbfgs", 0.0), ("liblinear", 1.0))
    for peso in (None, "balanced")
    for C in (0.01, 10.0)
]

medicoes = pd.DataFrame([
    {**{k: v for k, v in caso.items() if k != "max_iter"},
     **{k: v for k, v in medir_ajuste(matriz_treino, alvo_treino, caso).items()
        if k in ("segundos", "n_iter", "convergiu", "coeficientes_nulos")}}
    for caso in piores_casos
])
medicoes.sort_values("segundos", ascending=False)

O resultado desmonta duas suposições que pareciam seguras:

| solver | `C` | `class_weight` | tempo | iterações | converge |
|---|---:|---|---:|---:|---|
| `lbfgs` | 10.0 | `balanced` | 76,9 s | 838 | **não, com `max_iter=800`** |
| `lbfgs` | 0.01 | `None` | 72,2 s | 794 | sim |
| `lbfgs` | 10.0 | `None` | 42,1 s | 508 | sim |
| `lbfgs` | 0.01 | `balanced` | 36,9 s | 469 | sim |
| `liblinear` | 0.01 | `None` | 7,8 s | 20 | sim |
| `liblinear` | 0.01 | `balanced` | 5,0 s | 21 | sim |
| `liblinear` | 10.0 | `None` | 1,4 s | 15 | sim |
| `liblinear` | 10.0 | `balanced` | 1,1 s | 14 | sim |

**Primeira: o limite de 800 iterações herdado do card #206 não cobre a grade.** A combinação
`C=10.0` com `class_weight="balanced"` precisa de **838 iterações**, então com 800 ela truncaria e
entregaria coeficiente provisório para o card #212 ler como odds ratio. O `max_iter` da grade sobe
para **1600**, o degrau seguinte da mesma escala que o #206 usou. Subir o teto não encarece a
busca: `max_iter` é limite, não trabalho fixo, e o `lbfgs` para quando converge. As combinações que
já convergiam em 469 ou 508 iterações continuam custando o mesmo.

**Segunda: o tempo não cresce com `C`.** Sem reponderação o extremo caro é `C=0.01` (72,2 s contra
42,1 s), e com `balanced` a ordem se inverte (36,9 s contra 76,9 s). Escolher um extremo por
intuição erraria a conta por quase o dobro, e sempre é possível errar para menos, que é o lado que
promete uma busca que não cabe na sessão.

**Terceira, que não era suposição e sim descoberta útil:** o `liblinear` é de uma ordem de grandeza
mais barato. A metade L1 da grade custa menos do que um único ajuste da metade L2.

In [ ]:
custo = custo_estimado(SEGUNDOS_POR_AJUSTE, n_folds=N_FOLDS)

print(f"{custo['combinacoes']} combinações x {custo['n_folds']} folds + refit = {custo['ajustes']} ajustes")
for solver, segundos in custo["por_solver"].items():
    print(f"  {solver:>10}: {segundos / 60:5.1f} min")
print(f"  {'refit':>10}: {custo['refit'] / 60:5.1f} min")
print(f"\ntotal: {custo['segundos'] / 60:.1f} min em série, "
      f"{custo['segundos'] / 60 / 2:.1f} min com os 2 núcleos do Colab (n_jobs=-1)")

**A busca cabe na sessão do Colab.** São 81 ajustes: 51,3 min na metade `lbfgs`, 5,2 min na metade
`liblinear` e 1,3 min de `refit`, ou **57,7 min em série**. Com `n_jobs=-1` nos dois núcleos que o
Colab entrega, cerca de **29 min**, bem dentro do limite de inatividade de 90 min.

A conta é teto por dois motivos, os dois a favor: cada ajuste da busca roda sobre quatro quintos do
treino, porque o quinto restante é o fold de validação, e a grade tem combinações muito mais
baratas do que o pior caso usado na multiplicação.

**O que este card entrega ao card #210**, que é quem executa a busca:

- a grade em `src/espaco_busca_logistica.py`, importável, com `grade_com_prefixo()` para o nome de
  passo do `Pipeline` do card #208;
- `max_iter=1600` fixo, medido contra o pior caso e não herdado;
- a conta de custo acima, para a execução ser comparada com o previsto em vez de descoberta no
  meio.

O mesmo critério de dimensionamento vai para a Luiza, que declara a grade da Árvore de Decisão:
medir os extremos do espaço, não o ponto de partida da biblioteca.

## 3. Pipeline sobre o contrato (card #208)

As Seções 1 e 2 mediram o custo de um ajuste e declararam a grade. Falta o objeto que a busca do
card #210 vai receber: **um único `Pipeline`** que encadeia o pré-processador do contrato e a
`LogisticRegression`.

Enquanto os dois vivem soltos, quem ajusta decide onde ajustar, e a escolha mais cômoda é
justamente a que vaza. As Seções 1 e 2 mediram sobre `preparo["matrizes"]["treino"]`, que já saiu
do pré-processador ajustado no treino inteiro. Para *medir tempo* isso é correto e proposital: o
número que aqueles cards queriam era o relógio do ajuste, sem a codificação das categóricas no
meio. Para *buscar hiperparâmetro* deixa de ser: a mediana da imputação e os quartis do
`RobustScaler` passariam a carregar as linhas que cada fold usa como validação, e a métrica de
validação subiria sem que nada tivesse melhorado.

Dentro de um `Pipeline`, essa escolha deixa de existir. O `GridSearchCV` chama `fit` no objeto
inteiro a cada fold, e o `ColumnTransformer` é reajustado ali, sobre os quatro quintos daquele
fold. Não é uma disciplina que alguém precisa lembrar de seguir: é o que o card #209 trava por
teste.

### 3.1. O que o pipeline encadeia, e o que ele não decide

O pré-processador **não é remontado aqui**. `criar_pipeline` recebe o objeto que
`preparar_matriz` devolveu em `preparo["preprocessador"]` e faz `clone` dele: mesma especificação
(imputação, `RobustScaler`, `OneHotEncoder`, mesmas listas de colunas numéricas e categóricas),
sem o estado ajustado. Remontar o `ColumnTransformer` nesta célula criaria uma segunda definição
do contrato, e duas definições divergem sem ninguém perceber.

O `clone` também descarta de propósito o ajuste que `preparar_matriz` já fez no treino: o pipeline
precisa nascer virgem para que o primeiro `fit` seja o do fold.

Dois valores ficam fixos, e nenhum dos dois é escolha de hiperparâmetro deste card:

| Parâmetro | Valor | Por que aqui |
|---|---|---|
| `max_iter` | 1600 | Vem de `espaco_busca_logistica.MAX_ITER`, medido na Seção 2.3 contra o pior caso da grade. No padrão da biblioteca (100) o ajuste trunca, e coeficiente truncado é o que o card #212 leria como odds ratio |
| `random_state` | 42 | Mesma semente de `congelamento.SEMENTE`. O `liblinear` embaralha internamente: sem ela, duas execuções da mesma combinação dariam coeficientes diferentes |

`C`, `l1_ratio` e `class_weight` ficam no padrão da biblioteca. Quem escolhe os valores é a busca
do card #210 sobre a grade do card #207, não esta seção.

In [ ]:
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import ParameterGrid
from sklearn.utils.validation import check_is_fitted

from espaco_busca_logistica import grade_com_prefixo
from pipeline_logistica import (PASSO_MODELO, PASSO_PREPARO, ajustar_no_treino,
                                criar_pipeline)

pipeline = criar_pipeline(preparo["preprocessador"])
estimador = pipeline.named_steps[PASSO_MODELO]

print("passos:", [nome for nome, _ in pipeline.steps])
print(f"max_iter={estimador.max_iter}, random_state={estimador.random_state}, "
      f"C={estimador.C}, l1_ratio={estimador.l1_ratio}, class_weight={estimador.class_weight}")

# O pré-processador do pipeline tem que nascer NÃO ajustado, senão o primeiro
# `fit` de cada fold não seria o que define a mediana e a escala daquele fold.
try:
    check_is_fitted(pipeline.named_steps[PASSO_PREPARO])
    print("\nATENÇÃO: o pré-processador do pipeline já nasceu ajustado")
except NotFittedError:
    print("\npré-processador do pipeline: não ajustado, como esperado")

# A grade do card #207 endereça o passo do pipeline sem ser reescrita à mão.
combinacoes_prefixadas = list(ParameterGrid(grade_com_prefixo(PASSO_MODELO)))
print(f"\n{len(combinacoes_prefixadas)} combinações prefixadas, prontas para o card #210")
print("exemplo:", combinacoes_prefixadas[0])

O pipeline tem os dois passos, `preparo` e `modelo`, e o pré-processador nasce não ajustado. Os
hiperparâmetros saem no padrão da biblioteca (`C=1.0`, `l1_ratio=0.0`, `class_weight=None`),
como este card pede.

A última verificação é a que amarra este card ao #210: `grade_com_prefixo(PASSO_MODELO)` devolve
as mesmas **16 combinações** da Seção 2.1, agora com as chaves no formato `modelo__parâmetro`, que
é o único que o `GridSearchCV` reconhece quando o estimador é um `Pipeline`. O nome do passo vem
da constante `PASSO_MODELO`, não de uma string repetida nos dois arquivos: trocar o nome em um
lugar só faria a busca varrer um parâmetro inexistente, e o erro só apareceria no primeiro ajuste,
depois de a busca já ter começado.

### 3.2. Ajuste único sobre a partição de treino

Agora o pipeline inteiro é ajustado uma vez sobre `preparo["x"]["treino"]`, a matriz **crua**,
com as colunas da allowlist ainda por transformar, e não `preparo["matrizes"]["treino"]`, que já
passou pelo pré-processador nas Seções 1 e 2. É essa diferença que faz o `ColumnTransformer` ser
ajustado de dentro do pipeline.

O ajuste aqui não é a busca: é o ponto de partida contra o qual o card #210 vai comparar o ganho
do melhor conjunto de hiperparâmetros.

In [ ]:
relato_ajuste = ajustar_no_treino(pipeline, preparo["x"]["treino"], preparo["y"]["treino"])

for chave in ("n_linhas", "colunas_de_entrada", "colunas_da_matriz",
              "n_iter", "max_iter", "convergiu", "segundos", "coeficientes_nulos"):
    print(f"{chave}: {relato_ajuste[chave]}")

if relato_ajuste["aviso"]:
    print("\naviso de convergência:")
    print(relato_ajuste["aviso"])

O ajuste de dentro do pipeline **converge**, em **534 iterações** e algo em torno de **45 s**
(WSL2 local, CPU, `scikit-learn` 1.9.1; duas execuções deram 41,6 s e 50,4 s). São as mesmas 534
iterações que a Seção 1.5 mediu sobre a matriz já transformada, e a coincidência é o resultado que
se queria: encadear o pré-processador não muda o problema de otimização que o `lbfgs` resolve, só
muda **quando** a transformação acontece.

As **14 colunas de entrada** viram **38 colunas de matriz**: a expansão do `OneHotEncoder` sobre
as categóricas mais os indicadores de ausência do `SimpleImputer`, exatamente a largura que a
Seção 1.3 já reportava. Nenhum coeficiente sai zerado, o que era esperado com a penalidade L2 do
padrão da biblioteca: quem zera coeficiente é a L1, e ela entra na grade do card #210.

### 3.3. Métrica de partida: pendente do card #241

O card pede que a métrica de partida seja calculada com a função `avaliar()` do card 05A.1
(#241), e **não** com uma métrica escrita à parte. Esse é o ponto do card 05: as quatro duplas
comparam os candidatos na tabela do card 18A.1, e isso só funciona se todas lerem F2,
Sensibilidade, Precisão Média e ROC-AUC da mesma implementação.

O #241 ainda não está em `develop`, então a métrica não é registrada nesta MR. O que este card já
deixa pronto é o consumo: `pipeline_logistica.metrica_de_partida` recebe a função como argumento,
na assinatura acordada com o Arthur e o Cássio, e repassa os vetores sem calcular nem renomear
nada.

```python
from avaliacao import avaliar  # card 05A.1 (#241)

partida = metrica_de_partida(
    pipeline, preparo["x"]["validacao"], preparo["y"]["validacao"], avaliar
)
```

A probabilidade que entra em `avaliar` é a da classe positiva (Detrator), a coluna 1 de
`predict_proba`: Precisão Média e ROC-AUC são calculadas sobre o score contínuo, não sobre o
rótulo, e a coluna errada devolveria um número válido e errado.

A avaliação é sobre a **validação**, não sobre o teste. O teste é lido uma vez, na comparação
final do card 18A.1; gastá-lo aqui, antes da busca, transformaria a partição de teste numa segunda
validação.

## 4. Busca em grade da Regressão Logística (card #210)

A Seção 3 entregou o pipeline não ajustado. Esta seção passa esse objeto ao `GridSearchCV` e deixa
a busca escolher os hiperparâmetros que a Seção 2 declarou, sobre a validação cruzada por Cliente
do contrato.

Três travas valem para tudo o que segue, e nenhuma delas é escolha desta seção:

- **A validação é agrupada por `ID_GOLDENRECORD`**, e vem de `validacao.criar_folds`, não de um
  split montado aqui. O mesmo Cliente aparece em várias respostas, e um `StratifiedKFold` ingênuo
  colocaria o mesmo Cliente nos dois lados do fold: o modelo reconheceria o Cliente em vez de
  aprender o que leva à detração, e a métrica de validação subiria por vazamento.
- **A partição de teste não é tocada.** A busca recebe apenas `preparo["x"]["treino"]`. O teste
  fica para o card 13A.1 (#212), depois de os hiperparâmetros estarem escolhidos e congelados.
- **O pré-processamento acontece dentro de cada fold**, porque o que entra no `GridSearchCV` é o
  `Pipeline` da Seção 3, e não a matriz já transformada das Seções 1 e 2.

### 4.1. A grade reduzida, e por que ela foi reduzida

A Seção 2.3 estimou a grade inteira do card #207 em **58 minutos** de ajuste: 16
combinações vezes 5 folds, mais o refit, com o `lbfgs` respondendo por 51 desses minutos sozinho.
Não cabe na sessão de trabalho, e o card prevê o caso: reduzir a grade e registrar a redução, em
vez de interromper a execução no meio e reportar resultado parcial como final.

O corte é **em um eixo só**, o `C`, que vai de quatro valores para dois:

| Eixo | Card #207 | Aqui | Por quê |
|---|---|---|---|
| Penalidade | L2 (`lbfgs`) e L1 (`liblinear`) | as duas, intactas | É o eixo que o card #212 lê: a L1 zera coeficiente e a L2 não, e isso muda quais odds ratio existem para interpretar |
| `class_weight` | `None` e `balanced` | os dois, intactos | A prevalência de Detrator é 0,2043, e reponderar a classe é a decisão que mais mexe na Sensibilidade, que é o que o F2 privilegia |
| `C` | `[0.01, 0.1, 1.0, 10.0]` | `[0.1, 1.0]` | Restam os dois valores centrais. Os extremos varrem regularização muito forte e quase nenhuma, que são os cantos menos informativos para a leitura |

As 16 combinações viram **8**, e o custo estimado cai de 58 para cerca de 30 minutos de ajuste em série. A
grade é recortada a partir de `GRADE_LOGISTICA`, e não reescrita: reescrever criaria uma segunda
definição da grade, livre para divergir da do card #207 no dia em que ela mudar.

In [ ]:
from sklearn.metrics import fbeta_score, make_scorer
from sklearn.model_selection import GridSearchCV

from espaco_busca_logistica import GRADE_LOGISTICA, combinacoes
from validacao import N_FOLDS, criar_folds

# Recorte de um eixo só, a partir da grade do card #207.
C_REDUZIDO = [0.1, 1.0]
GRADE_REDUZIDA = [{**bloco, "C": C_REDUZIDO} for bloco in GRADE_LOGISTICA]

# Os folds vêm do contrato, agrupados por Cliente. Não se monta split aqui.
folds = criar_folds(preparo["x"]["treino"], preparo["grupos"]["treino"])

busca = GridSearchCV(
    criar_pipeline(preparo["preprocessador"]),
    param_grid=grade_com_prefixo(PASSO_MODELO, GRADE_REDUZIDA),
    scoring=make_scorer(fbeta_score, beta=2),  # provisório, ver 4.4
    cv=folds,
    refit=True,
    n_jobs=5,  # um processo por fold; não altera o resultado, só o relógio
)

n_comb = len(combinacoes(GRADE_REDUZIDA))
print(f"{n_comb} combinações x {N_FOLDS} folds = {n_comb * N_FOLDS} ajustes, mais o refit")

### 4.2. Execução da busca

O `fit` abaixo recebe a matriz **crua** do treino, e não a transformada: é o pipeline que
transforma, dentro de cada fold. O aviso de convergência é capturado em vez de silenciado, pelo
mesmo motivo da Seção 3.2: uma combinação que trunca em `max_iter` entrega coeficiente provisório,
e isso precisa aparecer.

In [ ]:
import time
import warnings

with warnings.catch_warnings(record=True) as capturados:
    warnings.simplefilter("always")
    inicio = time.perf_counter()
    busca.fit(preparo["x"]["treino"], preparo["y"]["treino"])
    segundos_busca = time.perf_counter() - inicio

print("melhores hiperparâmetros:")
for chave, valor in sorted(busca.best_params_.items()):
    print(f"  {chave.replace(PASSO_MODELO + '__', '')}: {valor}")

print(f"\nmelhor F2 médio nos folds: {busca.best_score_:.4f}")
print(f"tempo total da busca: {segundos_busca:.1f}s ({segundos_busca / 60:.1f} min)")
print(f"ajustes feitos: {len(busca.cv_results_['params']) * N_FOLDS} + refit")
print(f"avisos de convergência: "
      f"{sum('onvergence' in a.category.__name__ for a in capturados)}")

A busca varreu as **8 combinações** em **40 ajustes mais o refit**, sobre
341.962 linhas e 307.510 Clientes do treino, e levou
**579 s (9,6 min)** de relógio com `n_jobs=5`. 40 ajustes
é exatamente 8 combinações vezes 5 folds, que é a conferência que o card pede: nenhum fold foi
pulado e nenhuma combinação ficou de fora.

Os hiperparâmetros vencedores:

| Parâmetro | Valor |
|---|---|
| `C` | 1.0 |
| `class_weight` | 'balanced' |
| `solver` | `liblinear` |
| penalidade | L1 (`l1_ratio=1.0`) |
| `max_iter` | 1600 |

**Melhor F2 médio na validação cruzada por Cliente: 0,5189.**

Avisos de convergência: nenhum, todas as combinações convergiram dentro do max_iter de 1600.

Uma ressalva que já cabe aqui, porque qualifica o número que está sendo registrado: a diferença de
F2 entre as quatro melhores combinações é de 0,0003,
enquanto o desvio entre os cinco folds da vencedora é de 0,0035, cerca de dez
vezes maior. O vencedor é o vencedor, mas a margem está dentro do ruído: trocar de vencedor entre
essas quatro não mudaria o desempenho de forma perceptível.

A tabela completa das combinações, com o desvio entre folds e a leitura de qual eixo moveu a
métrica, é o card 07A.4 (#211). Aqui ficam registrados apenas os três números que este card pede:
os parâmetros vencedores, a métrica do vencedor e o tempo total.

### 4.3. O `scoring` desta busca é provisório, e o card #241 ainda não entrou

O card pede que o `scoring` da busca seja o do card 05A.1 (#241), e **não** uma métrica escrita à
parte. É o mesmo ponto que deixou a métrica de partida pendente na Seção 3.3: o #241 não está em
`develop`, e a branch do card 05 (#238) entrega até agora apenas o documento do protocolo, sem
código.

O que está registrado e acordado é a **escolha** da métrica: F2, no protocolo do #238. O que falta
é a implementação compartilhada. Por isso esta busca roda com
`make_scorer(fbeta_score, beta=2)` montado aqui, que é o F2 do `scikit-learn` no limiar padrão de
0,5.

**Consequência prática, para quem for reler:** se o `avaliar()` do #241 usar limiar diferente de
0,5, ou média diferente para a classe positiva, o ranking das 8 combinações pode mudar. Assim que
o #241 entrar, esta célula troca o `scoring` e a busca é reexecutada antes de o card 13A.1 (#212)
consumir os hiperparâmetros. O CR02 fica declarado como pendente nesta MR.

### 4.4. Dois níveis de `TIER_VIAGEM` que o treino nunca vê

Levantado ao revisar a MR do card #208, e vale registrar aqui porque é a busca que herda o efeito.

`AZUL ONE` e `DIAMANTE UNIQUE` só aparecem na base a partir de **2025-10-24**. O corte de
validação é 2025-07-01, então as **1.353 linhas** desses dois tiers caem inteiras na partição de
**teste**: nem treino nem validação têm uma única linha delas.

O `OneHotEncoder` do contrato é `handle_unknown="ignore"`, então:

- o pré-processador ajustado no treino aprende **5 níveis** de `TIER_VIAGEM`, não 7, e a matriz
  sai com 38 colunas;
- **nenhum fold desta busca vê esses dois níveis**, e portanto nenhum hiperparâmetro foi escolhido
  levando-os em conta;
- no teste, as 1.353 linhas atravessam com o bloco de `TIER_VIAGEM` inteiro zerado. Não levanta
  erro e não emite aviso, então isso não apareceria sozinho em lugar nenhum.

Não é defeito do contrato: é o comportamento correto para categoria não vista, e a alternativa,
`handle_unknown="error"`, quebraria a pontuação do teste. O que não pode acontecer é o card 13A.1
(#212) ler o coeficiente de `TIER_VIAGEM` como se ele cobrisse os sete níveis, e o card 07B.1
(#213) interpretar isso em termos da operação. São justamente os Clientes de topo.